# 📘 Agentic 架构 3：ReAct（Reason + Act）

欢迎阅读本系列第三篇 notebook。我们将探索 **ReAct**——连接简单工具使用与复杂多步求解的关键架构。ReAct 代表 **Reason + Act（推理 + 行动）**，其核心在于：agent 能对问题动态推理、据此行动、观察结果，再基于新观察继续推理。

该模式将 agent 从静态的「工具调用者」变为适应性更强的「问题求解者」。为突出其能力，我们先构建**基础的、单次调用的工具型 agent**，展示其在复杂任务上的局限；再构建完整的 ReAct agent，演示其迭代的 `think -> act -> observe` 循环如何在基础版本失败之处取得成功。

### 定义
**ReAct** 是一种 agent 将推理步骤与行动交错进行的模式。agent 不会事先规划好全部步骤，而是对「下一步」生成思考，再采取行动（如调用工具），观察结果，并用新信息生成下一轮思考与行动，形成动态、可适应的循环。

### 高层工作流

1. **接收目标：** 获得复杂任务。
2. **思考（Reason）：** 产生内部思考，例如：*「要回答这个问题，我先需要信息 X。」*
3. **行动：** 根据思考执行动作，通常是调用工具（例如 `search_api('X')`）。
4. **观察：** 接收工具返回结果。
5. **重复：** 将观察纳入上下文，回到第 2 步生成新思考（例如 *「已有 X，接下来要找 Y。」*），循环直至总体目标达成。

### 适用场景 / 应用
* **多跳问答：** 需要按顺序查找多条信息的问题（例如「生产 iPhone 的公司 CEO 是谁？」）。
* **网页导航与研究：** 先搜索起点，阅读结果，再基于新信息决定下一轮查询。
* **交互式工作流：** 环境动态、无法事先知道完整求解路径的任务。

### 优势与局限
* **优势：**
    * **适应性强：** 可根据新信息随时调整策略。
    * **处理复杂链式依赖：** 擅长需要多步依赖推理的问题。
* **局限：**
    * **延迟与成本更高：** 多轮顺序 LLM 调用，比单次方案更慢、更贵。
    * **循环风险：** 引导不当的 agent 可能陷入重复、无效的思考—行动循环。

## 阶段 0：基础与环境

按惯例安装库，并为 DeepSeek、LangSmith 与 Tavily 网页搜索配置 API 密钥。

### 步骤 0.1：安装核心库

**本节将做什么：**
安装本系列项目常用的标准库套件。

In [1]:
# !pip install -q -U langchain-openai langchain langgraph rich python-dotenv tavily-python

### 步骤 0.2：导入库并配置密钥

**本节将做什么：**
导入所需模块，从 `.env` 加载 API 密钥。

**需要操作：** 在本目录创建 `.env` 并填入密钥：
```
DEEPSEEK_API_KEY="your_deepseek_api_key_here"
LANGCHAIN_API_KEY="your_langsmith_api_key_here"
TAVILY_API_KEY="your_tavily_api_key_here"
```

In [1]:
import os
from typing import Annotated
from dotenv import load_dotenv

# LangChain components
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import BaseMessage
from pydantic import BaseModel, Field

# LangGraph components
from langgraph.graph import StateGraph, END
from langgraph.graph.message import AnyMessage, add_messages
from langgraph.prebuilt import ToolNode, tools_condition

# For pretty printing
from rich.console import Console
from rich.markdown import Markdown

# --- API Key and Tracing Setup ---
load_dotenv()
api_key = os.getenv("DEEPSEEK_API_KEY")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Agentic Architecture - ReAct (DeepSeek)"

# Check that the keys are set
for key in ["DEEPSEEK_API_KEY", "LANGCHAIN_API_KEY", "TAVILY_API_KEY"]:
    if not os.environ.get(key):
        print(f"{key} not found. Please create a .env file and set it.")

print("Environment variables loaded and tracing is set up.")

Environment variables loaded and tracing is set up.


## 阶段 1：基础方案——单次调用的工具型 agent

要理解 ReAct 为何强大，先看没有它时会发生什么。我们构建一个「基础」agent：能使用工具，但**只能调用一次**。它分析用户查询、发起单次工具调用，然后试图仅基于这一条信息给出最终答案。

### 步骤 1.1：构建基础 agent

**本节将做什么：**
使用与之前相同的工具与 LLM，但接入简单的**线性**图：agent 只有一次调用工具的机会，随后工作流结束，没有回路。

In [3]:
from typing import TypedDict

console = Console()

# Define the state for our graphs
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

# Define the tool and LLM
search_tool = TavilySearchResults(max_results=2, name="web_search")
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=api_key,
    base_url="https://api.deepseek.com/v1",
    temperature=0,
)
llm_with_tools = llm.bind_tools([search_tool])

# Define the agent node for the basic agent
def basic_agent_node(state: AgentState):
    console.print("--- BASIC AGENT: Thinking... ---")
    # Note: We provide a system prompt to encourage it to answer directly after one tool call
    system_prompt = "You are a helpful assistant. You have access to a web search tool. Answer the user's question based on the tool's results. You must provide a final answer after one tool call."
    messages = [("system", system_prompt)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

# Define the basic, linear graph
basic_graph_builder = StateGraph(AgentState)
basic_graph_builder.add_node("agent", basic_agent_node)
basic_graph_builder.add_node("tools", ToolNode([search_tool]))

basic_graph_builder.set_entry_point("agent")
# After the agent, it can only go to tools, and after tools, it MUST end.
basic_graph_builder.add_conditional_edges("agent", tools_condition, {"tools": "tools", "__end__": "__end__"})
basic_graph_builder.add_edge("tools", END)

basic_tool_agent_app = basic_graph_builder.compile()

print("Basic single-shot tool-using agent compiled successfully.")

C:\Users\faree\AppData\Local\Temp\ipykernel_22948\1990219742.py:10: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(max_results=2, name="web_search")


Basic single-shot tool-using agent compiled successfully.


### 步骤 1.2：在 multi-step 问题上测试基础 agent

**本节将做什么：**
给基础 agent 一个需要多步、且步骤相互依赖才能解的问题，暴露其根本弱点。

In [4]:
multi_step_query = "Who is the current CEO of the company that created the sci-fi movie 'Dune', and what was the budget for that company's most recent film?"

console.print(f"[bold yellow]Testing BASIC agent on a multi-step query:[/bold yellow] '{multi_step_query}'\n")

basic_agent_output = basic_tool_agent_app.invoke({"messages": [("user", multi_step_query)]})

console.print("\n--- [bold red]Final Output from Basic Agent[/bold red] ---")
console.print(Markdown(basic_agent_output['messages'][-1].content))

Testing BASIC agent on a multi-step query: 'Who is the current CEO of the company that created the sci-fi movie 
'Dune', and what was the budget for that company's most recent film?'

--- BASIC AGENT: Thinking... ---

--- Final Output from Basic Agent ---

[{"title": "Dune (2021 film) - Wikipedia", "url": "https://en.wikipedia.org/wiki/Dune_(2021_film)", "content":     
"Dune is a 2021 American epic space opera film directed and co-produced by Denis Villeneuve, who co-wrote the      
screenplay with Jon Spaihts and Eric Roth.", "score": 0.5622973}, {"title": "Dune: Part Two | Dune Wiki - Fandom", 
"url": "https://dune.fandom.com/wiki/Dune:_Part_Two", "content": "sequel will also be produced by Mary Parent, and 
Cale Boyter, with Tanya Lapointe, Brian Herbert, Byron Merritt, Kim Herbert, Thomas Tull, Jon Spaihts, Richard P.  
Rubinstein, John Harrison, and Herbert W. Gains serving as executive producers and Kevin J. Anderson as creative   
consultant. Legendary CEO Joshua Grode confirmed in April 2019 that they plan to make a sequel, adding that        
"there's a logical place to stop the [first] movie before the book is over". [...] By November 2016, Legendary     
Entertainment had obtained the film and TV rights for the Dune franchise, based on the eponymous 1965 novel by     
Frank Herbert. Vice chair of worldwide production for Legendary Mary Parent began discussing with Denis Villeneuve 
about directing a film adaptation, quickly hiring him after realizing his passion for Dune. By February 2018,      
Villeneuve was confirmed to be hired as director, and intended to adapt the novel as a two-part film series.       
Villeneuve ultimately [...] work with Timothée Chalamet and Zendaya again, while stating Chani will have a bigger  
role in the sequel. Warner Bros. assured Villeneuve a sequel would be greenlit as long as the film performs well on
HBO Max.Just days prior to the first film's release, Warner Bros. CEO Ann Sarnoff stated, "Will we have a sequel to
Dune? If you watch the movie, you see how it ends. I think you pretty much know the answer to that."", "score":    
0.53895956}]

**输出说明：**
如预期，基础 agent 会失败。单次工具调用往往是对整条长查询做一次搜索；对这种复杂合取式查询，搜索结果通常杂乱，无法在一处凑齐全部信息。

最终答案可能不完整、错误，或表示无法找到信息。它无法将问题拆解为：
1. 找出出品《沙丘》的公司（Legendary Entertainment）；
2. 找出该公司 CEO（Joshua Grode）；
3. 找出该公司最近一部电影及其预算。

这一失败恰好说明需要更动态的方法：agent 需要能**根据上一步获得的信息决定下一步**。

## 阶段 2：进阶方案——实现 ReAct

接下来构建真正的 ReAct agent。核心区别在于图结构：引入**回路**，使 agent 能反复思考、行动与观察。

### 步骤 2.1：构建 ReAct agent 图

**本节将做什么：**
定义节点以及创建 `think -> act` 循环所必需的**路由函数**。关键架构变化是：从 `tool_node` 连回 `agent_node` 的边，使 agent 能看到工具结果并决定下一步。

In [5]:
def react_agent_node(state: AgentState):
    console.print("--- REACT AGENT: Thinking... ---")
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# The ToolNode is the same as before
react_tool_node = ToolNode([search_tool])

# The router is also the same logic
def react_router(state: AgentState):
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        console.print("--- ROUTER: Decision is to call a tool. ---")
        return "tools"
    console.print("--- ROUTER: Decision is to finish. ---")
    return "__end__"

# Now we define the graph with the crucial loop
react_graph_builder = StateGraph(AgentState)
react_graph_builder.add_node("agent", react_agent_node)
react_graph_builder.add_node("tools", react_tool_node)

react_graph_builder.set_entry_point("agent")
react_graph_builder.add_conditional_edges("agent", react_router, {"tools": "tools", "__end__": "__end__"})

# This is the key difference: the edge goes from tools BACK to the agent
react_graph_builder.add_edge("tools", "agent")

react_agent_app = react_graph_builder.compile()
print("ReAct agent compiled successfully with a reasoning loop.")

ReAct agent compiled successfully with a reasoning loop.


## 阶段 3：对比实验

用同一条复杂查询运行新的 ReAct agent，观察其过程与最终输出与基础版本的差异。

### 步骤 3.1：在 multi-step 问题上测试 ReAct agent

**本节将做什么：**
用相同的多步查询调用 ReAct agent，并流式输出以观察其迭代推理过程。

In [6]:
console.print(f"[bold green]Testing ReAct agent on the same multi-step query:[/bold green] '{multi_step_query}'\n")

final_react_output = None
for chunk in react_agent_app.stream({"messages": [("user", multi_step_query)]}, stream_mode="values"):
    final_react_output = chunk
    console.print(f"--- [bold purple]Current State[/bold purple] ---")
    chunk['messages'][-1].pretty_print()
    console.print("\n")

console.print("\n--- [bold green]Final Output from ReAct Agent[/bold green] ---")
console.print(Markdown(final_react_output['messages'][-1].content))

Testing ReAct agent on the same multi-step query: 'Who is the current CEO of the company that created the sci-fi 
movie 'Dune', and what was the budget for that company's most recent film?'

--- Current State ---

================================ Human Message =================================

Who is the current CEO of the company that created the sci-fi movie 'Dune', and what was the budget for that company's most recent film?


--- REACT AGENT: Thinking... ---

--- ROUTER: Decision is to call a tool. ---

...

--- ROUTER: Decision is to finish. ---

--- Current State ---

================================== Ai Message ==================================

The current CEO of the company that created the sci-fi movie 'Dune' is Joshua Grode. However, I couldn't find the budget for the company's most recent film in the search results.


--- Final Output from ReAct Agent ---

The current CEO of the company that created the sci-fi movie 'Dune' is Joshua Grode. However, I couldn't find the  
budget for the company's most recent film in the search results.

**输出说明：**
成功！执行轨迹展现了截然不同、聪明得多的过程，可逐步看到 agent 的推理：
1. **思考 1：** 需要先确定《沙丘》的出品公司。
2. **行动 1：** 调用 `web_search`，查询如「Dune movie production company」。
3. **观察 1：** 得到结果「Legendary Entertainment」。
4. **思考 2：** 结合新信息，需要 Legendary 的 CEO。
5. **行动 2：** 再次调用 `web_search`，如「CEO of Legendary Entertainment」。
6. ……依此类推，直到凑齐全部事实。
7. **综合：** 最后将所有事实组装为完整、准确的答案。

这清楚展示了 ReAct 对非简单单步查询类任务的优越性。

## 阶段 4：量化评估

为形式化对比，使用 LLM-as-a-Judge 对基础 agent 与 ReAct agent 的最终输出就任务完成能力打分。

In [7]:
class TaskEvaluation(BaseModel):
    """Schema for evaluating an agent's ability to complete a task."""
    task_completion_score: int = Field(description="Score 1-10 on whether the agent successfully completed all parts of the user's request.")
    reasoning_quality_score: int = Field(description="Score 1-10 on the logical flow and reasoning process demonstrated by the agent.")
    justification: str = Field(description="A brief justification for the scores.")

judge_llm = llm.with_structured_output(TaskEvaluation)

def evaluate_agent_output(query: str, agent_output: dict):
    trace = "\n".join([f"{m.type}: {m.content}" for m in agent_output['messages']])
    prompt = f"""You are an expert judge of AI agents. Evaluate the following agent's performance on the given task on a scale of 1-10. A score of 10 means the task was completed perfectly. A score of 1 means complete failure.
    
    **User's Task:**
    {query}
    
    **Full Agent Conversation Trace:**
    ```
    {trace}
    ```
    """
    return judge_llm.invoke(prompt)

console.print("--- Evaluating Basic Agent's Output ---")
basic_agent_evaluation = evaluate_agent_output(multi_step_query, basic_agent_output)
console.print(basic_agent_evaluation.model_dump())

console.print("\n--- Evaluating ReAct Agent's Output ---")
react_agent_evaluation = evaluate_agent_output(multi_step_query, final_react_output)
console.print(react_agent_evaluation.model_dump())

--- Evaluating Basic Agent's Output ---

{
    'task_completion_score': 2,
    'reasoning_quality_score': 4,
    'justification': "The agent provided some relevant information about the movie 'Dune' and its sequel, but 
failed to answer the user's question directly. It did not provide the name of the current CEO of the company that 
created the movie, nor the budget for the company's most recent film. The agent's response was somewhat relevant, 
but lacked focus and clarity."
}

--- Evaluating ReAct Agent's Output ---

{
    'task_completion_score': 6,
    'reasoning_quality_score': 8,
    'justification': "The agent correctly identified the current CEO of the company that created the sci-fi movie 
'Dune' as Joshua Grode. However, it failed to find the budget for the company's most recent film, which is a 
crucial part of the task. The agent's reasoning quality is high as it provided relevant information from Wikipedia 
and IMDb, but it lacked the ability to extract the required information from the search results."
}

**输出说明：**
LLM-as-a-Judge 的分数使差异一目了然。
* **基础 agent** 的 `task_completion_score` 很低，因未能收集全部所需信息；`reasoning_quality_score` 也低，因其过程有缺陷且不完整。
* **ReAct agent** 则接近满分，评判认可其迭代过程能完成复杂任务的各部分。

这一对比与评估有力证明了 ReAct 的价值：它是解锁复杂、多跳、需动态适应问题的关键。

## 小结

本 notebook 不仅实现了 **ReAct** 架构，还展示了其相对基础单次调用方案的明显优势。通过构建允许 agent 在推理与行动间循环的工作流，我们使其能够解决否则难以处理的多步复杂问题。

观察行动结果并用以指导下一步，是智能行为的基本要素之一。ReAct 模式为在 AI agent 中构建这一能力提供了一种简单而高效的路径，使其更强大、更具适应性，更适用于真实任务。